In [13]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

In [14]:
def eval_binary_model(name, y_test, y_prob, y_pred):
    print(f"\n=== {name} ===")
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))


In [15]:
results = []

def add_result(name, y_test, y_prob, y_pred):
    results.append({
        "Model": name,
        "ROC_AUC": roc_auc_score(y_test, y_prob),
        "Recall_Delayed": classification_report(y_test, y_pred, output_dict=True)["1"]["recall"],
        "Precision_Delayed": classification_report(y_test, y_pred, output_dict=True)["1"]["precision"]
    })

# Example usage after each run:
# add_result("Random Forest | TREE", y_test, y_prob, y_pred)

In [16]:
df_tree = pd.read_csv("model_ready_tree.csv")

TARGET = "delayed"  # change if your target column is named differently
X = df_tree.drop(columns=[TARGET])
y = df_tree[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [17]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_prob = rf.predict_proba(X_test)[:, 1]
y_pred = rf.predict(X_test)

eval_binary_model("Random Forest | TREE dataset", y_test, y_prob, y_pred)
add_result("Random Forest | TREE dataset", y_test, y_prob, y_pred)




=== Random Forest | TREE dataset ===
ROC-AUC: 0.7532695120924742
Confusion Matrix:
 [[20749 13365]
 [ 2134  6514]]
              precision    recall  f1-score   support

           0       0.91      0.61      0.73     34114
           1       0.33      0.75      0.46      8648

    accuracy                           0.64     42762
   macro avg       0.62      0.68      0.59     42762
weighted avg       0.79      0.64      0.67     42762



In [18]:
from xgboost import XGBClassifier

# scale_pos_weight = (neg/pos) helps imbalance
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / max(pos, 1)

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight
)

xgb.fit(X_train, y_train)

y_prob = xgb.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

eval_binary_model("XGBoost | TREE dataset", y_test, y_prob, y_pred)
add_result("XGBoost | TREE dataset", y_test, y_prob, y_pred)




=== XGBoost | TREE dataset ===
ROC-AUC: 0.8008388996853724
Confusion Matrix:
 [[24412  9702]
 [ 2291  6357]]
              precision    recall  f1-score   support

           0       0.91      0.72      0.80     34114
           1       0.40      0.74      0.51      8648

    accuracy                           0.72     42762
   macro avg       0.66      0.73      0.66     42762
weighted avg       0.81      0.72      0.74     42762



In [19]:
df_cb = pd.read_csv("model_ready_catboost.csv")

TARGET = "delayed"
X = df_cb.drop(columns=[TARGET])
y = df_cb[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [20]:
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
cat_features = [X_train.columns.get_loc(c) for c in cat_cols]


In [21]:
from catboost import CatBoostClassifier

# class weights based on imbalance
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
w0 = 1.0
w1 = neg / max(pos, 1)

cb = CatBoostClassifier(
    iterations=800,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    class_weights=[w0, w1]
)

cb.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

y_prob = cb.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

eval_binary_model("CatBoost | CatBoost dataset", y_test, y_prob, y_pred)
add_result("CatBoost | CatBoost dataset", y_test, y_prob, y_pred)


0:	test: 0.7267054	best: 0.7267054 (0)	total: 151ms	remaining: 2m
100:	test: 0.7906635	best: 0.7906635 (100)	total: 15.8s	remaining: 1m 49s
200:	test: 0.7983193	best: 0.7983193 (200)	total: 32.4s	remaining: 1m 36s
300:	test: 0.8030252	best: 0.8030252 (300)	total: 48.7s	remaining: 1m 20s
400:	test: 0.8051730	best: 0.8051730 (400)	total: 1m 5s	remaining: 1m 4s
500:	test: 0.8062692	best: 0.8062936 (499)	total: 1m 21s	remaining: 48.6s
600:	test: 0.8072509	best: 0.8072509 (600)	total: 1m 37s	remaining: 32.4s
700:	test: 0.8077791	best: 0.8078057 (692)	total: 1m 54s	remaining: 16.1s
799:	test: 0.8081157	best: 0.8081339 (769)	total: 2m 10s	remaining: 0us

bestTest = 0.8081339187
bestIteration = 769

Shrink model to first 770 iterations.

=== CatBoost | CatBoost dataset ===
ROC-AUC: 0.8081339187478106
Confusion Matrix:
 [[25421  8693]
 [ 2499  6149]]
              precision    recall  f1-score   support

           0       0.91      0.75      0.82     34114
           1       0.41      0.71    

In [22]:
df_nn = pd.read_csv("model_ready_nn.csv")

TARGET = "delayed"
X = df_nn.drop(columns=[TARGET])
y = df_nn[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [23]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=512,
    learning_rate_init=1e-3,
    max_iter=30,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

mlp.fit(X_train, y_train)

y_prob = mlp.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

eval_binary_model("Neural Net (MLP) | NN dataset", y_test, y_prob, y_pred)
add_result("Neural Net (MLP) | NN dataset", y_test, y_prob, y_pred)




=== Neural Net (MLP) | NN dataset ===
ROC-AUC: 0.7955949377195697
Confusion Matrix:
 [[32859  1255]
 [ 6348  2300]]
              precision    recall  f1-score   support

           0       0.84      0.96      0.90     34114
           1       0.65      0.27      0.38      8648

    accuracy                           0.82     42762
   macro avg       0.74      0.61      0.64     42762
weighted avg       0.80      0.82      0.79     42762



In [24]:
pd.DataFrame(results).sort_values("ROC_AUC", ascending=False)


,Model,ROC_AUC,Recall_Delayed,Precision_Delayed
2,CatBoost | CatBoost dataset,0.808134,0.711031,0.414297
1,XGBoost | TREE dataset,0.800839,0.735083,0.395853
3,Neural Net (MLP) | NN dataset,0.795595,0.265957,0.646976
0,Random Forest | TREE dataset,0.753270,0.753238,0.327682
